**Imports and File Checking**


In [1]:
!pip install -q transformers peft bitsandbytes accelerate gradio
import os
from google.colab import drive
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00


In [2]:
# Mount Drive
drive.mount('/content/drive')

# Check both models
classifier_path = "/content/drive/MyDrive/finalClassifier"
therapist_path = "/content/drive/MyDrive/Qwen2.5-3B-therapist/final"

print("Sarcasm classifier files:")
print(os.listdir(classifier_path))

print("\nTherapist LoRA files:")
print(os.listdir(therapist_path))

print("✅ Ready")

Mounted at /content/drive
Sarcasm classifier files:
['config.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'model.safetensors', 'EVAL.md']

Therapist LoRA files:
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']
✅ Ready


**Loading Sarcasm Classifier**
- This is the DistilBERT Model

In [3]:
SARCASM_PATH = "/content/drive/MyDrive/finalClassifier"

sarcasm_detector = pipeline(
    "text-classification",
    model=SARCASM_PATH,
    device=0  # use GPU
)

# Quick test
test = sarcasm_detector("Oh great, another wonderful day")
print(f"Test result: {test}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Test result: [{'label': 'LABEL_1', 'score': 0.5956559777259827}]


**Looading the Fine-Tuned Qwen Model**

In [4]:
BASE_MODEL   = "/content/drive/MyDrive/Qwen2.5-3B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/Qwen2.5-3B-therapist/final"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    local_files_only=True,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("✅ Tokenizer loaded")

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    local_files_only=True,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)
print("✅ Base model loaded")

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("✅ Fine-tuned model ready")

Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


✅ Tokenizer loaded
Loading base model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Base model loaded
Loading LoRA adapter...
✅ Fine-tuned model ready


**Integration Logic**

In [5]:
def generate_response(user_input):
    # Step 1 — Detect sarcasm
    sarcasm_result = sarcasm_detector(user_input)[0]
    is_sarcastic   = sarcasm_result["label"] == "LABEL_1"
    confidence     = sarcasm_result["score"]

    # Step 2 — Adjust system prompt based on sarcasm
    if is_sarcastic:
        system_prompt = """You are a compassionate mental health support assistant.
        The user is expressing themselves sarcastically, which may indicate frustration
        or emotional distress. Acknowledge their feelings gently and respond with empathy,
        reading between the lines of what they are really feeling."""
    else:
        system_prompt = """You are a compassionate mental health support assistant.
        Respond with empathy, warmth and understanding. Provide supportive and
        thoughtful responses that help the user feel heard and validated."""

    # Step 3 — Generate response with fine-tuned Qwen
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_input}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response, is_sarcastic, confidence

**Testing**

In [6]:
test_inputs = [
    "I feel so alone and nobody understands me",
    "Oh great, therapy is definitely going to fix all my problems",
    "I've been really anxious lately and don't know what to do",
    "Yeah sure, everything is just absolutely wonderful"
]

print("=== Pipeline Test ===\n")
for text in test_inputs:
    response, is_sarcastic, confidence = generate_response(text)
    label = "🙄 Sarcastic" if is_sarcastic else "💬 Genuine"
    print(f"Input:    {text}")
    print(f"Detected: {label} ({confidence:.2f})")
    print(f"Response: {response}")
    print("-" * 60)

=== Pipeline Test ===

Input:    I feel so alone and nobody understands me
Detected: 💬 Genuine (0.60)
Response: I understand how you feel. I think what you're experiencing is called isolation. Isolation can be defined as "a feeling of loneliness and separation from other people." It's important to know that you are not alone. Many people experience feelings of loneliness and isolation at some point in their lives. It's a normal human emotion. It's also important to know that you are not alone in your thoughts or feelings. We all experience loneliness, even if we don't always feel like it. Even if you do not feel like you need professional help, it may still be helpful to talk about your feelings with someone you trust. If you ever feel like you want to talk more about this or anything else, please reach out. I'm here for you.
------------------------------------------------------------
Input:    Oh great, therapy is definitely going to fix all my problems
Detected: 🙄 Sarcastic (0.97)
R

**Gradio Chat UI**

In [7]:
import gradio as gr

def chat(message, history):
    response, is_sarcastic, confidence = generate_response(message)

    # Add sarcasm indicator to response
    if is_sarcastic:
        indicator = f"*(Sarcasm detected: {confidence:.0%} confidence)*\n\n"
    else:
        indicator = ""

    return indicator + response

gr.ChatInterface(
    chat,
    title="🧠 AI Emotional Support Assistant",
    description="A compassionate AI chatbot with sarcasm detection. Type how you're feeling.",
    examples=[
        "I've been feeling really anxious lately",
        "Oh sure, because my life couldn't get any better",
        "I don't know how to cope with everything going on",
        "Yeah therapy is totally working great for me"
    ],
    theme=gr.themes.Soft()
).launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9b7e7be58f7a2875fb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
